In [1]:
import os
import pandas as pd 
import numpy as np 

In [2]:
# Read each file into Data folder
names = ['Concept_Flow','Amortization','CashFlow']

dfs = {}

for idx, file in enumerate(sorted(os.listdir('Data'))):
    if file.endswith('.csv'):
        dfs[names[idx]] = pd.read_csv(os.path.join('Data', file))
        print(f'{file} has been read successfully.')

dim_conceptos_flujo.csv has been read successfully.
fact_amortizacion.csv has been read successfully.
fact_flujo_caja.csv has been read successfully.


In [3]:
for data in dfs:
    print(f'{data} columns: {dfs[data].columns.size}')
    print(f'{data} rows: {dfs[data].shape[0]}')
    dfs[data].columns = dfs[data].columns.str.strip().str.upper() 
    for col in dfs[data].columns:
        print(f'{data} column {col} has {dfs[data][col].isnull().sum()} null values')
        
        if col.endswith('ID'):
            pass
            print(f'{data} {col} unique values in {col}: {dfs[data][col].nunique()}')
        elif col.startswith('FECHA'):
            print(f'{data} {col} type: {dfs[data][col].dtype}')

Concept_Flow columns: 4
Concept_Flow rows: 11
Concept_Flow column CONCEPTO_ID has 0 null values
Concept_Flow CONCEPTO_ID unique values in CONCEPTO_ID: 11
Concept_Flow column TIPO has 0 null values
Concept_Flow column CATEGORIA has 0 null values
Concept_Flow column SUBCATEGORIA has 0 null values
Amortization columns: 8
Amortization rows: 52
Amortization column CREDITO_ID has 0 null values
Amortization CREDITO_ID unique values in CREDITO_ID: 3
Amortization column ENTIDAD has 0 null values
Amortization column NUMERO_CUOTA has 0 null values
Amortization column FECHA_VENCIMIENTO has 0 null values
Amortization FECHA_VENCIMIENTO type: str
Amortization column CUOTA_TOTAL has 0 null values
Amortization column ABONO_CAPITAL has 0 null values
Amortization column PAGO_INTERES has 0 null values
Amortization column SALDO_PENDIENTE has 0 null values
CashFlow columns: 3
CashFlow rows: 264
CashFlow column FECHA has 0 null values
CashFlow FECHA type: str
CashFlow column CONCEPTO_ID has 0 null values
Cas

In [4]:
# Merge the dataframe Concept with CashFlow 

Columns = ['FECHA', 'CONCEPTO_ID', 'MONTO','TIPO','CATEGORIA','SUBCATEGORIA']
ConceptFlow = dfs['Concept_Flow']
CashFlow = dfs['CashFlow']
MergeTable = CashFlow.merge(ConceptFlow, on='CONCEPTO_ID', how='left')[Columns]



In [ ]:
# View the CashFlow

MergeTable['Monto_Original'] = np.where(MergeTable['TIPO'] == 'Ingreso', MergeTable['MONTO'], -MergeTable['MONTO'])




print(MergeTable.head(5))

        FECHA  CONCEPTO_ID    MONTO     TIPO     CATEGORIA  \
0  2024-06-01          101  32000.0  Ingreso     Operativo   
1  2024-06-01          102  15000.0  Ingreso     Operativo   
2  2024-06-01          103   8000.0  Ingreso  No Operativo   
3  2024-06-01          104    450.0  Ingreso  No Operativo   
4  2024-06-01          201  22000.0   Egreso     Operativo   

                SUBCATEGORIA  Monto_Original  
0  SaaS Monthly Subscription         32000.0  
1     SaaS Enterprise Annual         15000.0  
2  Consultoria Especializada          8000.0  
3   Rendimientos Financieros           450.0  
4           Nomina Core Team        -22000.0  


In [9]:
#CashFlowByDay = MergeTable.groupby('FECHA').agg({'Monto_Original': 'sum'}).reset_index()

OperaProfit =   (MergeTable[MergeTable['CATEGORIA'] == 'Operativo']
                .groupby(['FECHA','TIPO','CATEGORIA'])
                .agg({'Monto_Original': 'sum'}).reset_index())

OperaProfit.to_csv('OperaProfit.csv', index=False)
MergeTable.to_csv('MergeTable.csv', index=False)

In [10]:
#Create a proyection of the credit amortization 

proyection = dfs['Amortization'].groupby('FECHA_VENCIMIENTO').agg({
    'CUOTA_TOTAL': 'sum'
    ,'PAGO_INTERES': 'sum'
    ,'ABONO_CAPITAL': 'sum'
    ,'SALDO_PENDIENTE': 'sum'
    }).reset_index()

proyection.to_csv('proyeccion.csv', index=False)  

print(proyection.head(10))

  FECHA_VENCIMIENTO  CUOTA_TOTAL  PAGO_INTERES  ABONO_CAPITAL  SALDO_PENDIENTE
0        2025-01-01      5992.39       1800.00        4192.39        115807.61
1        2025-02-01      5992.39       1737.11        4255.28        111552.33
2        2025-03-01      5992.39       1673.28        4319.11        107233.22
3        2025-04-01      5992.39       1608.49        4383.90        102849.32
4        2025-05-01      5992.39       1542.73        4449.66         98399.66
5        2025-06-01     10963.60       2435.99        8527.61        169872.05
6        2025-07-01     10963.60       2320.11        8643.50        161228.55
7        2025-08-01     10963.60       2202.63        8760.97        152467.58
8        2025-09-01     10963.60       2083.54        8880.06        143587.52
9        2025-10-01     13316.50       2562.81       10753.69        182833.83
